# Heat Demand Analysis — EU Pulp & Paper Facilities (2024)

This notebook explores results from `calculate_heat_demand.py`:

- **Interactive map** of facilities colored by classification, sized by heat demand
- **Aggregations** by country and by product classification
- **Validation checks** against published production data and Eurostat sector benchmarks

Run `python3 heat_demand/calculate_heat_demand.py` first to refresh `heat_demand_by_facility.csv`.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Support running from project root or heat_demand/
_cwd = Path.cwd()
if (_cwd / "heat_demand" / "heat_demand_by_facility.csv").exists():
    PROJECT_ROOT = _cwd
elif (_cwd / "heat_demand_by_facility.csv").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd.parent

HEAT_DEMAND_PATH = PROJECT_ROOT / "heat_demand" / "heat_demand_by_facility.csv"
CLASSIFICATION_PATH = PROJECT_ROOT / "Input" / "pulp_paper_fossil_classification.xlsx"

MWH_TO_TJ = 0.0036  # 1 MWh_th = 3.6 GJ = 0.0036 TJ

CLASSIFICATION_COLORS = {
    "Pulp": "#1f77b4",
    "Integrated": "#ff7f0e",
    "Paper/Board": "#2ca02c",
    "Tissue": "#9467bd",
}

BAT_INTENSITY_RANGES_GJ_T = {
    "Pulp": (10, 16),
    "Integrated": (14, 20),
    "Paper/Board": (4, 6),
    "Tissue": (7, 12),
}

In [5]:
heat = pd.read_csv(HEAT_DEMAND_PATH)
meta = pd.read_excel(CLASSIFICATION_PATH, sheet_name="Facilities", header=1, engine="openpyxl")
fossil_shares = pd.read_excel(CLASSIFICATION_PATH, sheet_name="Fossil_Shares", header=1, engine="openpyxl")
eu_benchmark = pd.read_excel(CLASSIFICATION_PATH, sheet_name="EU_Benchmark_Verify", header=0, engine="openpyxl")

df = heat.merge(
    meta[[
        "source_id", "source_type", "justification", "annual_emissions_co2e_t",
        "total_TJ_country", "eurostat_country",
    ]],
    on="source_id",
    how="left",
)

df["heat_demand_tj"] = df["heat_demand_mwh_th"] * MWH_TO_TJ
df["replaceable_heat_tj"] = df["replaceable_heat_mwh_th"] * MWH_TO_TJ
df["implied_intensity_gj_per_t"] = (
    df["heat_demand_mwh_th"] * 3.6 / df["annual_output_t"]
)
df["has_coordinates"] = df["lat"].notna() & df["lon"].notna()

print(f"Facilities loaded: {len(df)}")
print(f"With coordinates: {df['has_coordinates'].sum()}")
print(f"With replaceable heat: {df['replaceable_heat_mwh_th'].notna().sum()}")
df.head()

Facilities loaded: 88
With coordinates: 78
With replaceable heat: 76


,source_id,source_name,iso3_corrected,country_corrected,classification,confidence,nace_used,annual_output_t,heat_intensity_mwh_per_t,fossil_share_country,...,status_flag,source_type,justification,annual_emissions_co2e_t,total_TJ_country,eurostat_country,heat_demand_tj,replaceable_heat_tj,implied_intensity_gj_per_t,has_coordinates
0,44375328,Laakirchen Papier AG Pulp Mill,AUT,Austria,Paper/Board,High,C17xP,260039.19997,1.3,0.0000,...,NaN,Pulp misc.,Recycled-fibre SC paper & containerboard; no v...,151082.77519,9197.390,Austria,1216.983456,0.000000,4.68,True
1,44375330,Mondi Neusiedler Pulp Mill,AUT,Austria,Integrated,High,C17,260039.19997,5.0,0.2809,...,NaN,Chemical wood pulp; Mechanical and semi-chemic...,Kematen mill: sulphite pulp + uncoated fine pa...,151082.77519,61834.396,Austria,4680.705599,1314.810203,18.00,True
2,44375331,Papierfabrik Wattens GmbH & Co KG - Delfort,AUT,Austria,Paper/Board,Medium,C17xP,260039.19997,1.3,0.0000,...,NEEDS_VERIFICATION,Semi-chemical wood pulp,Delfort specialty/thin paper mill; semi-chem t...,151082.77519,9197.390,Austria,1216.983456,0.000000,4.68,True
3,44375537,SCA Laakirchen Papier AG,AUT,Austria,Paper/Board,Medium,C17xP,485789.47367,1.3,0.0000,...,DUPLICATE|COUNTRY_CORRECTED,Pulp misc.,"COUNTRY CORRECTED to AUT (Laakirchen, Austria)...",282243.68419,9197.390,Austria,2273.494737,0.000000,4.68,True
4,44375329,UPM-Kymmene Steyrermuhl Austria Pulp Mill,AUT,Austria,Paper/Board,High,C17xP,260039.19997,1.3,0.0000,...,NaN,Chemical wood pulp,Graphic paper mill; converted to Heinzel packa...,151082.77519,9197.390,Austria,1216.983456,0.000000,4.68,True


## Portfolio summary

In [6]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Facilities",
            "Countries",
            "Total annual output (Mt)",
            "Total heat demand (TWh_th)",
            "Total replaceable heat (TWh_th)",
            "Avg fossil share (weighted)",
        ],
        "Value": [
            len(df),
            df["country_corrected"].nunique(),
            df["annual_output_t"].sum() / 1e6,
            df["heat_demand_mwh_th"].sum() / 1e6,
            df["replaceable_heat_mwh_th"].sum(skipna=True) / 1e6,
            df["replaceable_heat_mwh_th"].sum(skipna=True) / df["heat_demand_mwh_th"].sum(),
        ],
    }
)
summary

,Metric,Value
0,Facilities,88.000000
1,Countries,19.000000
2,Total annual output (Mt),30.577090
3,Total heat demand (TWh_th),101.654553
4,Total replaceable heat (TWh_th),17.735876
5,Avg fossil share (weighted),0.174472


## Analysis by country

In [7]:
by_country = (
    df.groupby("country_corrected", as_index=False)
    .agg(
        sites=("source_id", "count"),
        annual_output_mt=("annual_output_t", lambda s: s.sum() / 1e6),
        heat_demand_twh=("heat_demand_mwh_th", lambda s: s.sum() / 1e6),
        replaceable_heat_twh=("replaceable_heat_mwh_th", lambda s: s.sum(skipna=True) / 1e6),
        heat_demand_tj=("heat_demand_tj", "sum"),
        replaceable_heat_tj=("replaceable_heat_tj", lambda s: s.sum(skipna=True)),
        avg_fossil_share=("fossil_share_country", "mean"),
    )
    .sort_values("heat_demand_twh", ascending=False)
)
by_country["replaceable_share"] = (
    by_country["replaceable_heat_twh"] / by_country["heat_demand_twh"]
)
by_country

,country_corrected,sites,annual_output_mt,heat_demand_twh,replaceable_heat_twh,heat_demand_tj,replaceable_heat_tj,avg_fossil_share,replaceable_share
16,Sweden,18,8.024417,30.576602,0.678530,110075.765976,2442.707797,0.023244,0.022191
12,Portugal,9,3.950164,14.339869,2.163113,51623.529279,7787.208562,0.184611,0.150846
4,France,10,3.859373,12.663824,3.167811,45589.765562,11404.118059,0.272870,0.250146
5,Germany,14,3.266492,8.089831,2.720210,29123.391506,9792.755314,0.361429,0.336251
11,Poland,4,1.998419,7.806217,2.248308,28102.382121,8093.908651,0.289350,0.288015
8,Italy,3,1.749213,5.305696,2.801335,19100.504373,10084.807636,0.493033,0.527986
15,Spain,3,1.118850,4.961188,2.542815,17860.277375,9154.132827,0.567200,0.512541
3,Czechia,3,1.238122,4.358698,0.000000,15691.311515,0.000000,NaN,0.000000
0,Austria,6,1.785985,3.986032,0.708373,14349.715183,2550.142025,0.101800,0.177714
1,Belgium,2,0.581140,2.905701,0.113903,10460.524028,410.052542,0.039200,0.039200


In [8]:
fig_country = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Heat demand by country (TWh_th)", "Replaceable heat by country (TWh_th)"),
)
fig_country.add_trace(
    go.Bar(
        x=by_country["country_corrected"],
        y=by_country["heat_demand_twh"],
        marker_color="#4C78A8",
        name="Heat demand",
    ),
    row=1, col=1,
)
fig_country.add_trace(
    go.Bar(
        x=by_country["country_corrected"],
        y=by_country["replaceable_heat_twh"],
        marker_color="#F58518",
        name="Replaceable heat",
    ),
    row=1, col=2,
)
fig_country.update_layout(
    height=500, width=1100, showlegend=False,
    title="Country-level heat demand summary",
)
fig_country.update_xaxes(tickangle=-45)
# fig_country.show()

## Analysis by classification

In [9]:
by_class = (
    df.groupby("classification", as_index=False)
    .agg(
        sites=("source_id", "count"),
        annual_output_mt=("annual_output_t", lambda s: s.sum() / 1e6),
        heat_demand_twh=("heat_demand_mwh_th", lambda s: s.sum() / 1e6),
        replaceable_heat_twh=("replaceable_heat_mwh_th", lambda s: s.sum(skipna=True) / 1e6),
        avg_intensity_mwh_per_t=("heat_intensity_mwh_per_t", "first"),
        avg_fossil_share=("fossil_share_country", "mean"),
    )
    .sort_values("heat_demand_twh", ascending=False)
)
by_class

,classification,sites,annual_output_mt,heat_demand_twh,replaceable_heat_twh,avg_intensity_mwh_per_t,avg_fossil_share
2,Pulp,32,11.452552,45.810210,4.124455,4.0,0.122444
0,Integrated,25,7.800651,39.003256,8.851387,5.0,0.243996
1,Paper/Board,28,9.693089,12.601016,3.035996,1.3,0.269004
3,Tissue,3,1.630797,4.240071,1.724037,2.6,0.412833


In [10]:
fig_class = px.bar(
    by_class,
    x="classification",
    y=["heat_demand_twh", "replaceable_heat_twh"],
    barmode="group",
    color_discrete_sequence=["#4C78A8", "#F58518"],
    labels={
        "value": "TWh_th",
        "classification": "Product classification",
        "variable": "Metric",
    },
    title="Heat demand and replaceable heat by classification",
    category_orders={"classification": list(CLASSIFICATION_COLORS.keys())},
)
fig_class.update_layout(height=450, width=800)
fig_class.show()

In [11]:
fig_pie = px.pie(
    by_class,
    names="classification",
    values="heat_demand_twh",
    color="classification",
    color_discrete_map=CLASSIFICATION_COLORS,
    title="Share of total heat demand by classification",
    hole=0.35,
)
fig_pie.update_layout(height=450, width=700)
fig_pie.show()

## Interactive map — heat demand per site

Marker **color** = product classification. Marker **size** = total heat demand (MWh_th/yr). Hover for site details.

In [12]:
map_df = df[df["has_coordinates"]].copy()
map_df["marker_size"] = np.clip(
    map_df["heat_demand_mwh_th"] / map_df["heat_demand_mwh_th"].max() * 40 + 8,
    8, 48,
)

map_df["hover_text"] = (
    "<b>" + map_df["source_name"] + "</b><br>"
    + "Country: " + map_df["country_corrected"] + " (" + map_df["iso3_corrected"] + ")<br>"
    + "Classification: " + map_df["classification"] + "<br>"
    + "Confidence: " + map_df["confidence"].fillna("—") + "<br>"
    + "NACE: " + map_df["nace_used"] + "<br>"
    + "Annual output: " + map_df["annual_output_t"].map(lambda x: f"{x:,.0f} t/yr") + "<br>"
    + "Heat intensity: " + map_df["heat_intensity_mwh_per_t"].astype(str) + " MWh_th/t<br>"
    + "Heat demand: " + map_df["heat_demand_mwh_th"].map(lambda x: f"{x:,.0f} MWh_th/yr") + "<br>"
    + "Fossil share: " + map_df["fossil_share_country"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "N/A") + "<br>"
    + "Replaceable heat: " + map_df["replaceable_heat_mwh_th"].map(
        lambda x: f"{x:,.0f} MWh_th/yr" if pd.notna(x) else "N/A"
    )
)

fig_map = px.scatter_mapbox(
    map_df,
    lat="lat",
    lon="lon",
    color="classification",
    color_discrete_map=CLASSIFICATION_COLORS,
    size="marker_size",
    size_max=48,
    hover_name="source_name",
    custom_data=["hover_text"],
    zoom=3,
    center={"lat": 54, "lon": 12},
    mapbox_style="carto-positron",
    title="EU pulp & paper facilities — heat demand by site (2024)",
    category_orders={"classification": list(CLASSIFICATION_COLORS.keys())},
)
fig_map.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
)
fig_map.update_layout(height=700, width=1100, margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig_map.show()

/var/folders/7b/k9t7lbq9461cdn3czk_fmvxm0000gn/T/ipykernel_8900/2405402667.py:22: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(


## Validation against published data

### 1. ENCE pulp mills (Spain) — production check

Reported 2020 production from [ENCE disclosures](https://ence.es/en/pulp-business/) vs Climate TRACE 2024 estimates.

In [13]:
ence_published = pd.DataFrame(
    {
        "site": ["ENCE Navia", "ENCE Pontevedra"],
        "reported_production_t_yr": [572_567, 434_718],
        "reported_year": [2020, 2020],
        "source": "ENCE annual report / pulp business page",
    }
)

ence_estimated = df[df["source_name"].str.contains("ENCE", case=False, na=False)][[
    "source_name", "annual_output_t", "heat_demand_mwh_th", "classification",
    "heat_intensity_mwh_per_t", "fossil_share_country", "replaceable_heat_mwh_th",
]].copy()
ence_estimated["site"] = ence_estimated["source_name"].str.extract(
    r"(Navia|Pontevedra)", expand=False
)
ence_estimated["site"] = "ENCE " + ence_estimated["site"]

ence_validation = ence_published.merge(
    ence_estimated,
    on="site",
    how="left",
)
ence_validation["production_diff_pct"] = (
    (ence_validation["annual_output_t"] - ence_validation["reported_production_t_yr"])
    / ence_validation["reported_production_t_yr"] * 100
)
ence_validation[[
    "site", "reported_year", "reported_production_t_yr", "annual_output_t",
    "production_diff_pct", "heat_demand_mwh_th", "replaceable_heat_mwh_th",
]]

,site,reported_year,reported_production_t_yr,annual_output_t,production_diff_pct,heat_demand_mwh_th,replaceable_heat_mwh_th
0,ENCE Navia,2020,572567,361371.86293,-36.885664,1.445487e+06,988424.319486
1,ENCE Pontevedra,2020,434718,271688.33489,-37.502396,1.086753e+06,743121.933591


### 2. Heat intensity vs BAT reference ranges

Implied intensity = heat demand / annual output. Compare to BAT BREF ranges from the README (GJ/t).

In [14]:
intensity_check = df[[
    "source_name", "country_corrected", "classification",
    "implied_intensity_gj_per_t", "heat_intensity_mwh_per_t",
]].copy()
intensity_check["bat_low_gj_per_t"] = intensity_check["classification"].map(
    lambda c: BAT_INTENSITY_RANGES_GJ_T[c][0]
)
intensity_check["bat_high_gj_per_t"] = intensity_check["classification"].map(
    lambda c: BAT_INTENSITY_RANGES_GJ_T[c][1]
)
intensity_check["within_bat_range"] = (
    (intensity_check["implied_intensity_gj_per_t"] >= intensity_check["bat_low_gj_per_t"])
    & (intensity_check["implied_intensity_gj_per_t"] <= intensity_check["bat_high_gj_per_t"])
)

print(
    "Facilities within BAT range:",
    intensity_check["within_bat_range"].sum(),
    f"/ {len(intensity_check)} (fixed SEC applied — expect all match chosen values)",
)
intensity_check.groupby("classification")["implied_intensity_gj_per_t"].agg(["mean", "min", "max"])

Facilities within BAT range: 88 / 88 (fixed SEC applied — expect all match chosen values)


,mean,min,max
classification,,,
Integrated,18.00,18.00,18.00
Paper/Board,4.68,4.68,4.68
Pulp,14.40,14.40,14.40
Tissue,9.36,9.36,9.36


### 3. Country heat demand vs Eurostat sector energy (2023)

Compare summed facility heat demand (TJ) to Eurostat total final energy for the matching NACE sector.  
**Note:** Facility heat is modelled from production × SEC; Eurostat is reported final energy consumption — differences are expected but large gaps flag review.

In [15]:
NACE_SECTOR_MAP = {
    "C1711": "C1711 — Pulp only",
    "C17xP": "C17_X_C1711 — Paper excl. pulp",
    "C17": "C17 — All paper & pulp",
}

facility_by_country_nace = (
    df.groupby(["eurostat_country", "nace_used"], as_index=False)
    .agg(
        sites=("source_id", "count"),
        modelled_heat_tj=("heat_demand_tj", "sum"),
        modelled_replaceable_tj=("replaceable_heat_tj", lambda s: s.sum(skipna=True)),
    )
)
facility_by_country_nace["sector"] = facility_by_country_nace["nace_used"].map(NACE_SECTOR_MAP)

eurostat = fossil_shares.rename(columns={
    "Country": "eurostat_country",
    "Total TJ (NCV)": "eurostat_total_tj",
    "Fossil Share": "eurostat_fossil_share",
})[["eurostat_country", "Sector", "eurostat_total_tj", "eurostat_fossil_share", "Data Available"]]

country_validation = facility_by_country_nace.merge(
    eurostat,
    left_on=["eurostat_country", "sector"],
    right_on=["eurostat_country", "Sector"],
    how="left",
).drop(columns=["Sector"])

country_validation["modelled_vs_eurostat_pct"] = (
    country_validation["modelled_heat_tj"] / country_validation["eurostat_total_tj"] * 100
)
country_validation = country_validation.sort_values("modelled_vs_eurostat_pct", ascending=False)
country_validation[[
    "eurostat_country", "nace_used", "sites", "modelled_heat_tj",
    "eurostat_total_tj", "modelled_vs_eurostat_pct", "eurostat_fossil_share", "Data Available",
]]

,eurostat_country,nace_used,sites,modelled_heat_tj,eurostat_total_tj,modelled_vs_eurostat_pct,eurostat_fossil_share,Data Available
22,Poland,C1711,1,6697.163563,1550.094,432.048867,0.4339,Yes
33,Sweden,C1711,12,84183.160014,109186.827,77.100107,0.0203,Yes
19,Norway,C1711,2,2487.788800,3610.076,68.912366,0.1789,Yes
31,Spain,C1711,2,9116.066849,13425.626,67.900498,0.6838,Yes
2,Austria,C17xP,4,5924.445104,9197.390,64.414417,0.0000,Yes
9,France,C1711,3,14178.257421,22116.286,64.107768,0.0000,Yes
26,Portugal,C17xP,2,5190.849777,8440.673,61.498056,0.4566,Yes
24,Portugal,C17,3,27610.827178,53802.848,51.318523,0.1405,Yes
3,Belgium,C17,2,10460.524028,22562.816,46.361784,0.0392,Yes
25,Portugal,C1711,4,18821.852324,45362.175,41.492394,0.0817,Yes


### 4. EU-level benchmark verification (from classification workbook)

In [16]:
eu_benchmark_display = eu_benchmark.rename(columns={
    eu_benchmark.columns[0]: "metric",
    eu_benchmark.columns[1]: "eurostat_published",
    eu_benchmark.columns[2]: "derived_from_file",
    eu_benchmark.columns[3]: "match",
    eu_benchmark.columns[4]: "notes",
})
eu_benchmark_display = eu_benchmark_display.iloc[1:].reset_index(drop=True)
eu_benchmark_display

,metric,eurostat_published,derived_from_file,match,notes
0,C17xP — Total final energy (sum of reported co...,"~728,000 TJ GCV (=728 PJ)","678,231 TJ NCV",~5–8% lower expected (NCV vs GCV),NCV file; EU27 suppressed — summed available c...
1,C17xP — Natural gas share,20.3% (147.3 PJ GCV),"21.7% (147,444 TJ NCV)",✓ Close match expected,Share ratio independent of GCV/NCV conversion
2,C17xP — Solid fossil share,1.8% (12.9 PJ GCV),"2.3% (15,730 TJ NCV)",✓ Close match expected,Small absolute; some countries suppressed
3,C17xP — Fossil share (gas+solid+mgas+oil),~24% (natgas 20.3% + solid 1.8% + oil ~2%),26.5%,✓ Within ±3pp expected,Aggregate fossil share for paper-excl-pulp
4,C1711 — Total final energy (sum of reported co...,"~430,000 TJ GCV (estimated from 70.3% renew. s...","420,051 TJ NCV",~5–10% lower expected (NCV vs GCV),"Many major pulp countries (FIN, SWE) have larg..."
5,C1711 — Renewables share,70.3% (302.5 PJ GCV),~79.7% non-fossil (proxy),✓ High renewables expected,Renewables not in file; estimated as 1 - fossil
6,C1711 — Fossil share,~12–18% (complement of 70.3% renewables + ~12%...,20.3%,✓ Within range expected,"Includes nat gas, solid fossil, oil only (mgas..."


### 5. Data quality flags

Facilities flagged in the classification workbook for review (duplicates, mislabels, missing fossil share).

In [17]:
df["status_flag"] = df["status_flag"].fillna("")
flagged = df[
    (df["status_flag"] != "")
    | df["fossil_share_country"].isna()
    | ~df["has_coordinates"]
].copy()

flag_summary = pd.DataFrame({
    "Issue": [
        "Any status flag",
        "Missing fossil share",
        "Missing coordinates",
        "Modelled heat > Eurostat country total (same NACE)",
    ],
    "Count": [
        (df["status_flag"] != "").sum(),
        df["fossil_share_country"].isna().sum(),
        (~df["has_coordinates"]).sum(),
        (country_validation["modelled_vs_eurostat_pct"] > 100).sum(),
    ],
})
flag_summary

,Issue,Count
0,Any status flag,42
1,Missing fossil share,12
2,Missing coordinates,10
3,Modelled heat > Eurostat country total (same N...,1


In [18]:
review_cols = [
    "source_name", "country_corrected", "classification", "status_flag",
    "fossil_share_country", "annual_output_t", "heat_demand_mwh_th",
    "replaceable_heat_mwh_th", "justification",
]
flagged[review_cols].sort_values(["status_flag", "country_corrected"])

,source_name,country_corrected,classification,status_flag,fossil_share_country,annual_output_t,heat_demand_mwh_th,replaceable_heat_mwh_th,justification
9,Svilocell EAD - Subsidiary of Svilosa AD,Bulgaria,Pulp,,NaN,92820.826668,3.712833e+05,NaN,Bleached kraft market pulp; no integrated pape...
12,Biocel Paskov Pulp Mill,Czechia,Pulp,,NaN,219922.838700,8.796914e+05,NaN,Lenzing biorefinery: dissolving + paper-grade ...
19,Rayonier Advanced Materials,France,Pulp,,0.0000,13022.262480,5.208905e+04,0.000000e+00,RYAM Tartas: dissolving/high-purity cellulose ...
43,Smurfit Kappa Roermond Papier BV,Netherlands,Paper/Board,,NaN,175142.366290,2.276851e+05,NaN,Roermond: recycled containerboard/testliner; n...
8,Stambolijski Kraft Pulp & Paper Mill,Bulgaria,Integrated,CLOSED,NaN,92820.826668,4.641041e+05,NaN,Name states Pulp & Paper; historic kraft site ...
34,Sappi Stockstadt GmbH - Division of Sappi Fine...,Germany,Integrated,CLOSED,0.3814,164844.499980,8.242225e+05,3.143585e+05,Integrated 145kt internal pulp + 220kt CWF pap...
24,Arctic Paper Mochenwangen,Germany,Paper/Board,CLOSED|COUNTRY_CORRECTED,0.4110,798276.649510,1.037760e+06,4.265192e+05,"COUNTRY CORRECTED to DEU (Mochenwangen, Baden-..."
35,Sappi Stockstadt Mill,Germany,Integrated,CLOSED|DUPLICATE,0.3814,164844.499980,8.242225e+05,3.143585e+05,Duplicate of 44375514 — same integrated Stocks...
32,Palm GmbH & Co. KG,Germany,Paper/Board,COUNTRY_CORRECTED,0.4110,490081.309980,6.371057e+05,2.618504e+05,COUNTRY CORRECTED to DEU. Palm: recycled newsp...
44,Borregaard AS,Norway,Pulp,COUNTRY_CORRECTED,0.1789,148763.111100,5.950524e+05,1.064549e+05,"COUNTRY CORRECTED to NOR (Sarpsborg, Norway). ..."


## Export summary tables

In [19]:
output_dir = PROJECT_ROOT / "heat_demand" / "analysis_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

by_country.to_csv(output_dir / "heat_demand_by_country.csv", index=False)
by_class.to_csv(output_dir / "heat_demand_by_classification.csv", index=False)
ence_validation.to_csv(output_dir / "validation_ence_production.csv", index=False)
country_validation.to_csv(output_dir / "validation_country_vs_eurostat.csv", index=False)
flagged[review_cols].to_csv(output_dir / "facilities_flagged_for_review.csv", index=False)

print(f"Summary tables written to {output_dir}")

Summary tables written to /Users/annayuen/Desktop/IECC/industrial_decarb/SEC_PPI_IHB_potential/heat_demand/analysis_outputs
